# Generalização entre sites de modelos de aprendizado de máquina para TDAH (ADHD-200)

Pipeline final: leave-one-site-out sem vazamento, seleção de hiperparâmetros por validação cruzada aninhada, inferência por bootstrap e permutação intra-site, e explicabilidade dos fenótipos.

**Como executar:** rode as células em ordem. Ajuste apenas `BASE` na seção 2.
**Modalidades:** fenótipos; fMRI (força do ROI, 954 ROIs, AAL); fusões fenótipos + fMRI (954 ROIs e AAL).
**Famílias de teste:** confirmatória (fenótipos, força do ROI, 954 ROIs, fusão 954 ROIs; correção de Bonferroni ×4) e exploratória (AAL e fusão AAL; p sem correção).
**Cache:** as séries extraídas e as permutações ficam em `resultados_finais/cache`. Se a execução cair, rode de novo e ela retoma. Para recalcular do zero, apague essa pasta (obrigatório se mudar os dados, as grades ou o número de permutações).
**Permutações:** `N_PERM` na seção 2 (o padrão ignora 954 ROIs e a fusão com 954 ROIs, que são lentos; use um valor maior que zero para incluí-los).
**Saídas:** tabelas `.csv`, `resumo_final.json`, figuras 300 dpi em `figuras/` e `pacote_resultados.zip`.

In [ ]:
!pip install -q shap

## 1. Ambiente

In [ ]:
import os, re, glob, json, time, shutil, warnings
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from scipy.stats import norm, rankdata
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler

try:
    import shap
except ImportError:
    shap = None

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": 0.25})

## 2. Configuração

In [ ]:
BASE = "/content/drive/MyDrive/TCC-ADHD"
PHENO_DIR = f"{BASE}/treino"
FMRI_BASE = f"{BASE}/Dados_Oficiais"
OUT_DIR = f"{BASE}/resultados_finais"
CACHE_DIR = f"{OUT_DIR}/cache"
FIG_DIR = f"{OUT_DIR}/figuras"
for _d in (OUT_DIR, CACHE_DIR, FIG_DIR):
    os.makedirs(_d, exist_ok=True)

N_ROIS = 954
PREFIXO_AAL = "sfnwmrda"
THRESH = 0.5
PHENO_COLS = ["Gender", "Age", "Full4 IQ"]

CSVS = {
    "KKI": "KKI_phenotypic.csv", "NeuroIMAGE": "NeuroIMAGE_phenotypic.csv",
    "NYU": "NYU_phenotypic.csv", "OHSU": "OHSU_phenotypic.csv",
    "Peking_1": "Peking_1_phenotypic.csv", "Peking_2": "Peking_2_phenotypic.csv",
    "Peking_3": "Peking_3_phenotypic.csv", "Pittsburgh": "Pittsburgh_phenotypic.csv",
    "WashU": "WashU_phenotypic.csv",
}
FMRI_DIRS = {
    "KKI": f"{FMRI_BASE}/rois_1000_kki/rois",
    "NeuroIMAGE": f"{FMRI_BASE}/rois_1000_neuroimage/rois",
    "NYU": f"{FMRI_BASE}/rois_1000_nyu/rois",
    "OHSU": f"{FMRI_BASE}/rois_1000_ohsu/rois",
    "Peking_1": f"{FMRI_BASE}/rois_1000_beijing/rois",
    "Peking_2": f"{FMRI_BASE}/rois_1000_beijing/rois",
    "Peking_3": f"{FMRI_BASE}/rois_1000_beijing/rois",
    "Pittsburgh": f"{FMRI_BASE}/rois_1000_pittsburgh/rois",
    "WashU": f"{FMRI_BASE}/rois_1000_washu/rois",
}
SITE_GROUP = {"Peking_1": "Peking", "Peking_2": "Peking", "Peking_3": "Peking"}

MODALIDADES = {
    "pheno": dict(feat=None, pheno=True, rotulo="Fenótipos"),
    "forca": dict(feat="forca", pheno=False, rotulo="fMRI (força do ROI)"),
    "fmri": dict(feat="fmri", pheno=False, rotulo="fMRI (954 ROIs)"),
    "fusion": dict(feat="fmri", pheno=True, rotulo="Fenótipos + fMRI (954 ROIs)"),
    "aal": dict(feat="aal", pheno=False, rotulo="fMRI (AAL)"),
    "fusion_aal": dict(feat="aal", pheno=True, rotulo="Fenótipos + fMRI (AAL)"),
}
C_ALTA_DIM = [1e-4, 1e-3, 1e-2]
GRADES = {
    "pheno": dict(K=[None], C=[1e-2, 1e-1, 1.0, 10.0]),
    "forca": dict(K=[100, 400, 1908], C=C_ALTA_DIM),
    "fmri": dict(K=[200, 1000, 5000], C=C_ALTA_DIM),
    "fusion": dict(K=[200, 1000, 5000], C=C_ALTA_DIM),
    "aal": dict(K=[100, 500, 2000], C=C_ALTA_DIM),
    "fusion_aal": dict(K=[100, 500, 2000], C=C_ALTA_DIM),
}
N_PERM = {"pheno": 1000, "forca": 200, "fmri": 0, "fusion": 0, "aal": 200, "fusion_aal": 200}
N_BOOT = 2000
LOTE_PERM = 20
FAMILIA_CONFIRMATORIA = ["pheno", "forca", "fmri", "fusion"]

## 3. Fenótipos, controle de qualidade e amostra

In [ ]:
def resumo_amostra(df, titulo):
    """Tabela de controles e casos de TDAH por site, impressa e devolvida."""
    t = (df.groupby(["Grupo_Site", "Target"]).size().unstack(fill_value=0)
           .rename(columns={0: "Controle", 1: "TDAH"}))
    for c in ("Controle", "TDAH"):
        if c not in t:
            t[c] = 0
    t["Total"] = t["Controle"] + t["TDAH"]
    print(f"\n=== {titulo} (N={len(df)}) ===")
    print(t.to_string())
    return t


frames = []
for nome, csv in CSVS.items():
    caminho = os.path.join(PHENO_DIR, nome, csv)
    if not os.path.exists(caminho):
        print(f"[AVISO] CSV ausente: {caminho}")
        continue
    d = pd.read_csv(caminho)
    d["Sub_Site"] = nome
    d["Grupo_Site"] = SITE_GROUP.get(nome, nome)
    d["fMRI_dir"] = FMRI_DIRS[nome]
    frames.append(d)
df = pd.concat(frames, ignore_index=True)

id_cols = [c for c in ("ScanDir ID", "ScanDirID") if c in df.columns]
df["Patient_ID"] = df[id_cols].bfill(axis=1).iloc[:, 0]

dx_num = pd.to_numeric(df["DX"], errors="coerce")
dx_txt = df["DX"].astype(str).map({
    "ADHD-Combined": 1, "ADHD-Inattentive": 1, "ADHD-Hyperactive/Impulsive": 1,
    "Typically Developing": 0})
df["Target"] = np.where(dx_num.notna(), (dx_num > 0).astype(float), dx_txt)

df["Gender"] = df["Gender"].replace({"M": 1, "F": 0, "Male": 1, "Female": 0})
for c in PHENO_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df.loc[df[c] <= -999, c] = np.nan

amostra_bruta = resumo_amostra(df.dropna(subset=["Target"]), "Antes do controle de qualidade")

df = df.dropna(subset=["Target", "Patient_ID"])
df = df[df["QC_Rest_1"] == 1.0]
df = df.drop_duplicates(subset=["Patient_ID", "Sub_Site"]).reset_index(drop=True)
df["Patient_ID"] = df["Patient_ID"].astype(int)
df["Target"] = df["Target"].astype(int)

df_qc = df[["Patient_ID", "Sub_Site", "Grupo_Site", *PHENO_COLS, "Target", "fMRI_dir"]].copy()
amostra_qc = resumo_amostra(df_qc, "Após controle de qualidade")

classes_por_site = df_qc.groupby("Grupo_Site")["Target"].nunique()
sites_removidos = list(classes_por_site[classes_por_site < 2].index)
df_qc = df_qc[~df_qc["Grupo_Site"].isin(sites_removidos)].reset_index(drop=True)
print("\nSites removidos por terem uma só classe:", sites_removidos)
print("Valores ausentes por variável (imputados dentro de cada fold):")
print(df_qc[PHENO_COLS].isna().sum().to_string())

## 4. Séries temporais: atlas de 954 ROIs

In [ ]:
def indexar_arquivos(pasta):
    """Mapeia ID numérico -> caminho dos arquivos *_run1.mat, ignorando zeros à esquerda."""
    mapa = {}
    if os.path.isdir(pasta):
        for f in os.listdir(pasta):
            m = re.search(r"(\d+)_run1\.mat$", f)
            if m:
                mapa[int(m.group(1))] = os.path.join(pasta, f)
    return mapa


def extrair_conectividade(tabela, n_rois):
    """Correlação de Pearson com transformada z de Fisher (triângulo superior) por sujeito."""
    iu = np.triu_indices(n_rois, k=1)
    indices = {d: indexar_arquivos(d) for d in tabela["fMRI_dir"].unique()}
    X = np.empty((len(tabela), len(iu[0])), dtype=np.float32)
    mantidos, falhas = [], []
    for pos, row in tqdm(tabela.iterrows(), total=len(tabela), desc="fMRI 954 ROIs"):
        pid = int(row["Patient_ID"])
        caminho = indices[row["fMRI_dir"]].get(pid)
        if caminho is None:
            falhas.append((pid, row["Grupo_Site"], "arquivo_ausente"))
            continue
        try:
            ts = sio.loadmat(caminho)["tseries"]
        except Exception as e:
            falhas.append((pid, row["Grupo_Site"], f"leitura:{type(e).__name__}"))
            continue
        if ts.ndim != 2 or ts.shape[1] != n_rois:
            falhas.append((pid, row["Grupo_Site"], "formato"))
            continue
        with np.errstate(invalid="ignore", divide="ignore"):
            r = np.corrcoef(ts.T)
        X[len(mantidos)] = np.arctanh(np.clip(r[iu], -0.999999, 0.999999))
        mantidos.append(pos)
    return X[: len(mantidos)], mantidos, falhas, iu


arq_amostra = f"{CACHE_DIR}/amostra_final.csv"
arq_fmri = f"{CACHE_DIR}/X_fmri_bruto.npy"
if os.path.exists(arq_amostra) and os.path.exists(arq_fmri):
    df_ok = pd.read_csv(arq_amostra)
    X_fmri = np.load(arq_fmri)
    iu = np.triu_indices(N_ROIS, k=1)
    print("Carregado do cache:", X_fmri.shape)
else:
    X_fmri, mantidos, falhas, iu = extrair_conectividade(df_qc, N_ROIS)
    df_ok = df_qc.loc[mantidos].reset_index(drop=True)
    pd.DataFrame(falhas, columns=["Patient_ID", "Grupo_Site", "motivo"]).to_csv(
        f"{OUT_DIR}/falhas_extracao.csv", index=False)
    df_ok.to_csv(arq_amostra, index=False)
    np.save(arq_fmri, X_fmri)
    print(f"Processados: {len(mantidos)} | Falhas: {len(falhas)}")

amostra_final = resumo_amostra(df_ok, "Amostra final")
pd.concat({"bruta": amostra_bruta, "pos_QC": amostra_qc, "final": amostra_final}).to_csv(
    f"{OUT_DIR}/amostra_por_site.csv")

y_all = df_ok["Target"].values.astype(int)
groups = df_ok["Grupo_Site"].values
sites = list(np.unique(groups))
X_pheno = df_ok[PHENO_COLS].values.astype(np.float64)

## 5. Séries temporais: atlas AAL

In [ ]:
def ler_serie(caminho):
    """Lê um arquivo .1D numérico, com ou sem cabeçalho de texto."""
    try:
        return np.atleast_2d(np.loadtxt(caminho, comments="#")).astype(float)
    except Exception:
        d = pd.read_csv(caminho, sep=r"\s+", engine="python")
        return d.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all").values.astype(float)


def extrair_aal(tabela):
    """Conectividade (Pearson + Fisher z) do atlas AAL a partir dos arquivos .1D por sujeito."""
    series = []
    for _, row in tqdm(tabela.iterrows(), total=len(tabela), desc="fMRI AAL"):
        pasta = f"{BASE}/treino/{row['Sub_Site']}/{int(row['Patient_ID']):07d}"
        achados = sorted(glob.glob(f"{pasta}/{PREFIXO_AAL}*session_1_rest_1_aal_TCs.1D"))
        if not achados:
            raise FileNotFoundError(f"Série AAL ausente: {pasta}")
        series.append(ler_serie(achados[0]))
    n_reg = pd.Series([s.shape[1] for s in series]).mode().iloc[0]
    if any(s.shape[1] != n_reg for s in series):
        raise ValueError("Número de regiões AAL inconsistente entre sujeitos.")
    iu_a = np.triu_indices(n_reg, k=1)
    X = np.empty((len(series), len(iu_a[0])), dtype=np.float32)
    for i, s in enumerate(series):
        with np.errstate(invalid="ignore", divide="ignore"):
            r = np.corrcoef(s.T)
        X[i] = np.arctanh(np.clip(r[iu_a], -0.999999, 0.999999))
    return X


arq_aal = f"{CACHE_DIR}/X_aal_bruto.npy"
if os.path.exists(arq_aal):
    X_aal = np.load(arq_aal)
else:
    X_aal = extrair_aal(df_ok)
    np.save(arq_aal, X_aal)
print("X_aal:", X_aal.shape)

## 6. Harmonização por site e força do ROI

In [ ]:
def zscore_por_site(X, grupos, bloco=50_000):
    """Padroniza cada variável dentro de cada site, sem usar rótulos. Opera no próprio array."""
    for g in np.unique(grupos):
        linhas = np.where(grupos == g)[0]
        for s in range(0, X.shape[1], bloco):
            cols = np.arange(s, min(s + bloco, X.shape[1]))
            blk = X[np.ix_(linhas, cols)].astype(np.float64)
            mu, sd = np.nanmean(blk, axis=0), np.nanstd(blk, axis=0)
            sd[(sd == 0) | ~np.isfinite(sd)] = 1.0
            X[np.ix_(linhas, cols)] = ((blk - mu) / sd).astype(np.float32)
    return X


def forca_roi(X, n_rois, iu):
    """Média de |z| e de z por ROI (2 x n_rois variáveis por sujeito)."""
    saida = np.zeros((X.shape[0], 2 * n_rois), dtype=np.float32)
    for k in range(X.shape[0]):
        M = np.zeros((n_rois, n_rois), dtype=np.float32)
        M[iu] = np.nan_to_num(X[k], nan=0.0)
        M = M + M.T
        saida[k, :n_rois] = np.abs(M).sum(1) / (n_rois - 1)
        saida[k, n_rois:] = M.sum(1) / (n_rois - 1)
    return saida


X_forca = zscore_por_site(forca_roi(X_fmri, N_ROIS, iu), groups)
X_fmri = zscore_por_site(X_fmri, groups)
X_aal = zscore_por_site(X_aal, groups)
FEATS = {"forca": X_forca, "fmri": X_fmri, "aal": X_aal}
print({k: v.shape for k, v in FEATS.items()})

## 7. Núcleo: seleção univariada, regressão logística e CV aninhada

In [ ]:
def auc(y, p):
    """AUC pela estatística de Mann-Whitney (empates com ranks médios)."""
    r = rankdata(p)
    n1 = int(y.sum())
    n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def f_scores_binario(X, linhas, y_linhas, bloco=20_000):
    """F de ANOVA para duas classes (equivale a f_classif). Colunas com NaN no treino recebem -inf."""
    n = len(linhas)
    i1, i0 = linhas[y_linhas == 1], linhas[y_linhas == 0]
    n1, n0 = len(i1), len(i0)
    saida = np.empty(X.shape[1], dtype=np.float64)
    for s in range(0, X.shape[1], bloco):
        sl = slice(s, min(s + bloco, X.shape[1]))
        a, b = X[i1, sl].astype(np.float64), X[i0, sl].astype(np.float64)
        invalido = np.isnan(a).any(0) | np.isnan(b).any(0)
        m1, m0 = a.mean(0), b.mean(0)
        m = (a.sum(0) + b.sum(0)) / n
        ssb = n1 * (m1 - m) ** 2 + n0 * (m0 - m) ** 2
        ssw = ((a - m1) ** 2).sum(0) + ((b - m0) ** 2).sum(0)
        with np.errstate(divide="ignore", invalid="ignore"):
            f = ssb / (ssw / (n - 2))
        f[~np.isfinite(f)] = 0.0
        f[invalido] = -np.inf
        saida[sl] = f
    return saida


def montar(spec, X, Xp, tr, te, K, scores):
    """Monta as matrizes de treino e teste. Imputação, padronização e seleção usam só o treino."""
    parte_tr, parte_te = [], []
    if spec["pheno"]:
        imp = SimpleImputer(strategy="median", keep_empty_features=True)
        esc = StandardScaler()
        parte_tr.append(esc.fit_transform(imp.fit_transform(Xp[tr])))
        parte_te.append(esc.transform(imp.transform(Xp[te])))
    if spec["feat"] is not None:
        k = min(K, X.shape[1])
        sel = np.sort(np.argpartition(-scores, k - 1)[:k])
        parte_tr.append(X[np.ix_(tr, sel)])
        parte_te.append(np.nan_to_num(X[np.ix_(te, sel)], nan=0.0))
    return np.hstack(parte_tr), np.hstack(parte_te)


def ajustar(Xtr, ytr, C):
    """Regressão logística L2 com pesos de classe balanceados."""
    return LogisticRegression(C=C, class_weight="balanced", solver="lbfgs",
                              max_iter=1000, tol=1e-3).fit(Xtr, ytr)


def escolher_hiperparametros(y, spec, grade, X, Xp, grp, tr):
    """CV aninhada: leave-one-site-out interno apenas nos sites de treino; maximiza a AUC média por site."""
    resultados = {}
    for itr, ite in LeaveOneGroupOut().split(tr, y[tr], grp[tr]):
        a, b = tr[itr], tr[ite]
        if len(set(y[a])) < 2 or len(set(y[b])) < 2:
            continue
        scores = None if X is None else f_scores_binario(X, a, y[a])
        for K in grade["K"]:
            Xa, Xb = montar(spec, X, Xp, a, b, K, scores)
            for C in grade["C"]:
                p = ajustar(Xa, y[a], C).predict_proba(Xb)[:, 1]
                resultados.setdefault((K, C), []).append(auc(y[b], p))
    if not resultados:
        return grade["K"][0], grade["C"][0]
    medias = {k: np.mean(v) for k, v in resultados.items()}
    return max(medias, key=medias.get)


def avaliar_nucleo(y, spec, grade, X, Xp, grp):
    """Leave-one-site-out externo. Devolve probabilidades fora da amostra e os hiperparâmetros por site."""
    prob = np.zeros(len(y))
    escolhas = []
    for tr, te in LeaveOneGroupOut().split(Xp, y, grp):
        K, C = escolher_hiperparametros(y, spec, grade, X, Xp, grp, tr)
        scores = None if X is None else f_scores_binario(X, tr, y[tr])
        Xtr, Xte = montar(spec, X, Xp, tr, te, K, scores)
        prob[te] = ajustar(Xtr, y[tr], C).predict_proba(Xte)[:, 1]
        escolhas.append((grp[te][0], K, C))
    return prob, escolhas


def avaliar(y, modo, Xp=None):
    """Avalia uma modalidade com a matriz de variáveis correspondente."""
    spec = MODALIDADES[modo]
    X = FEATS[spec["feat"]] if spec["feat"] else None
    return avaliar_nucleo(y, spec, GRADES[modo], X, X_pheno if Xp is None else Xp, groups)


def metricas_por_site(y, prob):
    """Métricas por site e agregadas (AUC média entre sites, AUC pooled, acurácia balanceada)."""
    linhas = []
    for g in sites:
        m = groups == g
        yt, pr = y[m], prob[m]
        pred = (pr >= THRESH).astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, pred, labels=[0, 1]).ravel()
        sens = tp / (tp + fn) if (tp + fn) else np.nan
        espec = tn / (tn + fp) if (tn + fp) else np.nan
        linhas.append(dict(site=g, n=int(m.sum()), prev_TDAH=yt.mean(),
                           AUC=auc(yt, pr) if len(set(yt)) == 2 else np.nan,
                           ACC=(tp + tn) / len(yt), sens=sens, espec=espec,
                           BAL_ACC=np.nanmean([sens, espec])))
    tab = pd.DataFrame(linhas).set_index("site")
    pooled = dict(AUC_media_sites=tab["AUC"].mean(), AUC_dp_sites=tab["AUC"].std(),
                  AUC_pooled=auc(y, prob), BAL_ACC_media=tab["BAL_ACC"].mean(),
                  ACC_baseline=max(y.mean(), 1 - y.mean()))
    return tab, pooled


def auc_media_entre_sites(y, prob, grp):
    """Estatística usada no bootstrap e nas permutações: média das AUC por site."""
    return np.mean([auc(y[grp == g], prob[grp == g]) for g in np.unique(grp)])

## 8. Avaliação das seis modalidades

In [ ]:
PROBS, TABELAS, RESUMOS, ESCOLHAS = {}, {}, {}, {}
for modo, spec in MODALIDADES.items():
    t0 = time.time()
    PROBS[modo], ESCOLHAS[modo] = avaliar(y_all, modo)
    TABELAS[modo], RESUMOS[modo] = metricas_por_site(y_all, PROBS[modo])
    print(f"{spec['rotulo']:30s} AUC média entre sites = {RESUMOS[modo]['AUC_media_sites']:.3f}  [{time.time() - t0:.0f}s]")

hiper = pd.DataFrame([dict(modalidade=m, site_teste=s, K=K, C=C) for m, esc in ESCOLHAS.items() for s, K, C in esc])
hiper["C_na_borda"] = hiper.apply(
    lambda r: r["C"] in (min(GRADES[r["modalidade"]]["C"]), max(GRADES[r["modalidade"]]["C"])), axis=1)
hiper.to_csv(f"{OUT_DIR}/hiperparametros_escolhidos.csv", index=False)
pd.concat(TABELAS).round(4).to_csv(f"{OUT_DIR}/metricas_por_site.csv")
np.save(f"{CACHE_DIR}/probabilidades.npy", np.vstack([PROBS[m] for m in MODALIDADES]))

print("\nMétricas por site - fenótipos:")
print(TABELAS["pheno"].round(3).to_string())

## 9. Inferência: bootstrap e testes de permutação intra-site

In [ ]:
def perm_worker(i, spec, grade, X, Xp, grp, y):
    """Uma permutação dos rótulos dentro de cada site seguida da avaliação completa (inclui a CV aninhada)."""
    r = np.random.default_rng([SEED, i])
    yp = y.copy()
    for g in np.unique(grp):
        idx = np.where(grp == g)[0]
        yp[idx] = r.permutation(y[idx])
    prob, _ = avaliar_nucleo(yp, spec, grade, X, Xp, grp)
    return auc_media_entre_sites(yp, prob, grp)


def permutacoes(modo, n_perm):
    """Distribuição nula da AUC média entre sites, com checkpoint em disco a cada lote."""
    arq = f"{CACHE_DIR}/perm_{modo}.npy"
    nulo = np.load(arq) if os.path.exists(arq) else np.full(n_perm, np.nan)
    if len(nulo) != n_perm:
        nulo = np.concatenate([nulo, np.full(max(0, n_perm - len(nulo)), np.nan)])[:n_perm]
    pend = [i for i in range(n_perm) if np.isnan(nulo[i])]
    spec = MODALIDADES[modo]
    X = FEATS[spec["feat"]] if spec["feat"] else None
    for s in tqdm(range(0, len(pend), LOTE_PERM), desc=f"Permutações {modo}"):
        lote = pend[s:s + LOTE_PERM]
        nulo[lote] = Parallel(n_jobs=-1)(
            delayed(perm_worker)(i, spec, GRADES[modo], X, X_pheno, groups, y_all) for i in lote)
        np.save(arq, nulo)
    return nulo


NULOS = {m: permutacoes(m, n) for m, n in N_PERM.items() if n > 0}

rng = np.random.default_rng(SEED)
celulas = {g: (np.where((groups == g) & (y_all == 1))[0], np.where((groups == g) & (y_all == 0))[0]) for g in sites}
boot = {m: np.empty(N_BOOT) for m in MODALIDADES}
for b in tqdm(range(N_BOOT), desc="Bootstrap"):
    idx = {g: np.concatenate([rng.choice(p, len(p)), rng.choice(n, len(n))]) for g, (p, n) in celulas.items()}
    for m in MODALIDADES:
        boot[m][b] = np.mean([auc(y_all[i], PROBS[m][i]) for i in idx.values()])

linhas = []
for m, spec in MODALIDADES.items():
    obs = RESUMOS[m]["AUC_media_sites"]
    lo, hi = np.percentile(boot[m], [2.5, 97.5])
    d = boot[m] - boot["pheno"]
    nulo = NULOS.get(m)
    p = (np.sum(nulo >= obs) + 1) / (len(nulo) + 1) if nulo is not None else np.nan
    linhas.append(dict(
        modalidade=m, rotulo=spec["rotulo"], AUC_media_sites=obs, IC95_inf=lo, IC95_sup=hi,
        AUC_dp_sites=RESUMOS[m]["AUC_dp_sites"], AUC_pooled=RESUMOS[m]["AUC_pooled"],
        BAL_ACC=RESUMOS[m]["BAL_ACC_media"], delta_vs_fenotipos=obs - RESUMOS["pheno"]["AUC_media_sites"],
        delta_IC_inf=np.percentile(d, 2.5), delta_IC_sup=np.percentile(d, 97.5),
        n_perm=0 if nulo is None else len(nulo), nulo_media=np.nan if nulo is None else nulo.mean(),
        nulo_dp=np.nan if nulo is None else nulo.std(), p_bruto=p,
        p_bonferroni=min(1.0, len(FAMILIA_CONFIRMATORIA) * p) if (m in FAMILIA_CONFIRMATORIA and not np.isnan(p)) else np.nan,
        familia="confirmatória" if m in FAMILIA_CONFIRMATORIA else "exploratória"))
INFERENCIA = pd.DataFrame(linhas).set_index("modalidade")
INFERENCIA.round(4).to_csv(f"{OUT_DIR}/inferencia_modalidades.csv")
print(INFERENCIA[["rotulo", "AUC_media_sites", "IC95_inf", "IC95_sup", "delta_vs_fenotipos",
                  "delta_IC_inf", "delta_IC_sup", "n_perm", "p_bruto", "p_bonferroni"]].round(4).to_string())

## 10. Poder estatístico

In [ ]:
def d_de_cohen(a):
    """Diferença padronizada equivalente a uma AUC, sob normalidade com variâncias iguais."""
    return np.sqrt(2) * norm.ppf(a)


poder = []
for m, nulo in NULOS.items():
    minimo = nulo.mean() + (norm.ppf(0.95) + norm.ppf(0.80)) * nulo.std()
    poder.append(dict(modalidade=m, AUC_minima_detectavel=minimo, d_equivalente=d_de_cohen(minimo),
                      AUC_observada=RESUMOS[m]["AUC_media_sites"],
                      d_observado=d_de_cohen(RESUMOS[m]["AUC_media_sites"])))
PODER = pd.DataFrame(poder).set_index("modalidade")
PODER.round(4).to_csv(f"{OUT_DIR}/poder_estatistico.csv")
print(PODER.round(3).to_string())

## 11. Explicabilidade dos fenótipos: coeficientes, tamanhos de efeito e SHAP

In [ ]:
coef_linhas, sv_blocos, xz_blocos, ordem = [], [], [], []
spec_p = MODALIDADES["pheno"]
for tr, te in LeaveOneGroupOut().split(X_pheno, y_all, groups):
    _, C = escolher_hiperparametros(y_all, spec_p, GRADES["pheno"], None, X_pheno, groups, tr)
    Xtr, Xte = montar(spec_p, None, X_pheno, tr, te, None, None)
    modelo = ajustar(Xtr, y_all[tr], C)
    coef_linhas.append(dict(site_teste=groups[te][0], C=C, **dict(zip(PHENO_COLS, modelo.coef_[0]))))
    if shap is not None:
        sv = np.asarray(shap.LinearExplainer(modelo, Xtr).shap_values(Xte))
        sv = sv[..., 1] if sv.ndim == 3 else sv
    else:
        sv = modelo.coef_[0] * (Xte - Xtr.mean(0))
    sv_blocos.append(sv)
    xz_blocos.append(Xte)
    ordem.append(te)

COEF = pd.DataFrame(coef_linhas).set_index("site_teste")
SV, XZ, ordem = np.vstack(sv_blocos), np.vstack(xz_blocos), np.concatenate(ordem)
y_ordem = y_all[ordem]

EXPLIC = pd.DataFrame({
    "coef_medio": COEF[PHENO_COLS].mean(),
    "coef_dp": COEF[PHENO_COLS].std(),
    "odds_ratio_por_1DP": np.exp(COEF[PHENO_COLS].mean()),
    "folds_mesmo_sinal": [(np.sign(COEF[c]) == np.sign(COEF[c].mean())).sum() for c in PHENO_COLS],
    "mean_abs_shap": np.abs(SV).mean(0),
    "shap_medio_TDAH": [SV[y_ordem == 1, j].mean() for j in range(len(PHENO_COLS))],
    "shap_medio_controle": [SV[y_ordem == 0, j].mean() for j in range(len(PHENO_COLS))],
})
efeitos = []
for j, c in enumerate(PHENO_COLS):
    ds, ns = [], []
    for g in sites:
        m = groups == g
        a = X_pheno[m & (y_all == 1), j]
        b = X_pheno[m & (y_all == 0), j]
        a, b = a[~np.isnan(a)], b[~np.isnan(b)]
        if len(a) > 1 and len(b) > 1:
            sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
            ds.append((a.mean() - b.mean()) / sp if sp > 0 else np.nan)
            ns.append(len(a) + len(b))
    efeitos.append(dict(variavel=c, d_ponderado=np.average(ds, weights=ns), d_min=np.min(ds), d_max=np.max(ds)))
EXPLIC = EXPLIC.join(pd.DataFrame(efeitos).set_index("variavel"))
EXPLIC.round(4).to_csv(f"{OUT_DIR}/explicabilidade_fenotipos.csv")
COEF.round(4).to_csv(f"{OUT_DIR}/coeficientes_por_fold.csv")
print(EXPLIC.round(3).to_string())

## 12. Sensibilidade dos fenótipos (exploratória)

In [ ]:
SUBCONJUNTOS = {
    "todas": PHENO_COLS,
    "sem_QI": [c for c in PHENO_COLS if c != "Full4 IQ"],
    "so_QI": ["Full4 IQ"],
    "sem_idade": [c for c in PHENO_COLS if c != "Age"],
    "sem_sexo": [c for c in PHENO_COLS if c != "Gender"],
}
SENSIBILIDADE = {}
for nome, cols in SUBCONJUNTOS.items():
    Xp_sub = X_pheno[:, [PHENO_COLS.index(c) for c in cols]]
    prob_sub, _ = avaliar(y_all, "pheno", Xp=Xp_sub)
    _, SENSIBILIDADE[nome] = metricas_por_site(y_all, prob_sub)
SENS = pd.DataFrame(SENSIBILIDADE).T
SENS.round(4).to_csv(f"{OUT_DIR}/sensibilidade_fenotipos.csv")
print(SENS[["AUC_media_sites", "AUC_dp_sites", "AUC_pooled", "BAL_ACC_media"]].round(3).to_string())

## 13. Figuras

In [ ]:
def salvar(nome):
    """Salva a figura em 300 dpi na pasta de figuras e a exibe."""
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{nome}.png", dpi=300)
    plt.show()


rotulos = {m: s["rotulo"] for m, s in MODALIDADES.items()}
ordem_mod = list(MODALIDADES)

fig, ax = plt.subplots(figsize=(7.5, 4))
comp = amostra_final[["Controle", "TDAH"]]
comp.plot(kind="bar", stacked=True, ax=ax, color=["#90A4AE", "#E64A19"], rot=0)
ax.set_xlabel("Site"); ax.set_ylabel("Sujeitos"); ax.set_title("Composição da amostra final", fontweight="bold")
salvar("01_composicao_amostra")

fig, ax = plt.subplots(figsize=(8, 4.5))
vals = [INFERENCIA.loc[m, "AUC_media_sites"] for m in ordem_mod]
erro = [[INFERENCIA.loc[m, "AUC_media_sites"] - INFERENCIA.loc[m, "IC95_inf"] for m in ordem_mod],
        [INFERENCIA.loc[m, "IC95_sup"] - INFERENCIA.loc[m, "AUC_media_sites"] for m in ordem_mod]]
ax.bar([rotulos[m] for m in ordem_mod], vals, yerr=erro, capsize=4, color="#1E88E5", alpha=0.85)
ax.axhline(0.5, color="gray", ls="--", label="Acaso (0,5)")
ax.set_ylim(0.35, 0.85); ax.set_ylabel("AUC média entre sites (IC95%)")
ax.set_title("Desempenho por modalidade (leave-one-site-out)", fontweight="bold")
plt.xticks(rotation=20, ha="right"); ax.legend(fontsize=8)
salvar("02_auc_por_modalidade")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
alvo = [m for m in ordem_mod if m != "pheno"]
dif = [INFERENCIA.loc[m, "delta_vs_fenotipos"] for m in alvo]
err = [[INFERENCIA.loc[m, "delta_vs_fenotipos"] - INFERENCIA.loc[m, "delta_IC_inf"] for m in alvo],
       [INFERENCIA.loc[m, "delta_IC_sup"] - INFERENCIA.loc[m, "delta_vs_fenotipos"] for m in alvo]]
ax.errorbar(dif, range(len(alvo)), xerr=err, fmt="o", color="#C62828", capsize=4)
ax.axvline(0, color="k", lw=0.8)
ax.set_yticks(range(len(alvo))); ax.set_yticklabels([rotulos[m] for m in alvo])
ax.set_xlabel("Diferença de AUC em relação aos fenótipos (IC95%)")
ax.set_title("Valor incremental do fMRI", fontweight="bold")
salvar("03_diferenca_vs_fenotipos")

mapa = pd.DataFrame({rotulos[m]: TABELAS[m]["AUC"] for m in ordem_mod})
fig, ax = plt.subplots(figsize=(8, 3.8))
im = ax.imshow(mapa.values, cmap="RdYlBu_r", vmin=0.3, vmax=0.8, aspect="auto")
ax.set_xticks(range(mapa.shape[1])); ax.set_xticklabels(mapa.columns, rotation=25, ha="right")
ax.set_yticks(range(mapa.shape[0])); ax.set_yticklabels(mapa.index); ax.grid(False)
for (i, j), v in np.ndenumerate(mapa.values):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, label="AUC")
ax.set_title("AUC por site e modalidade", fontweight="bold")
salvar("04_auc_site_modalidade")

if NULOS:
    fig, eixos = plt.subplots(1, len(NULOS), figsize=(4.2 * len(NULOS), 3.6), squeeze=False)
    for ax, (m, nulo) in zip(eixos[0], NULOS.items()):
        ax.hist(nulo, bins=25, color="lightgray", edgecolor="black")
        ax.axvline(RESUMOS[m]["AUC_media_sites"], color="green", lw=2.5)
        ax.set_title(f"{rotulos[m]}\np={INFERENCIA.loc[m, 'p_bruto']:.4f} (N={len(nulo)})", fontsize=9)
        ax.set_xlabel("AUC média entre sites")
    salvar("05_permutacoes")

fig, ax = plt.subplots(figsize=(6.2, 5.2))
for g in sites:
    m = groups == g
    fpr, tpr, _ = roc_curve(y_all[m], PROBS["pheno"][m])
    ax.plot(fpr, tpr, lw=1.4, label=f"{g} (AUC={TABELAS['pheno'].loc[g, 'AUC']:.2f})")
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("Taxa de falsos positivos"); ax.set_ylabel("Taxa de verdadeiros positivos")
ax.set_title("Curva ROC por site: fenótipos", fontweight="bold"); ax.legend(fontsize=8)
salvar("06_roc_fenotipos")

cm = confusion_matrix(y_all, (PROBS["pheno"] >= THRESH).astype(int))
fig, ax = plt.subplots(figsize=(4.8, 4.4))
ax.imshow(cm, cmap="Blues"); ax.grid(False)
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center", fontsize=15, fontweight="bold",
            color="white" if v > cm.max() / 2 else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Controle", "TDAH"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Controle", "TDAH"])
ax.set_xlabel("Previsão"); ax.set_ylabel("Diagnóstico real")
ax.set_title("Matriz de confusão: fenótipos", fontweight="bold")
salvar("07_matriz_confusao_fenotipos")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(PHENO_COLS, EXPLIC["coef_medio"], xerr=EXPLIC["coef_dp"], color="#43A047", capsize=4)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("Coeficiente padronizado (média ± DP entre folds)")
ax.set_title("Coeficientes da regressão logística", fontweight="bold")
salvar("08_coeficientes_fenotipos")

ordem_shap = EXPLIC["mean_abs_shap"].sort_values().index.tolist()
fig, eixos = plt.subplots(1, 2, figsize=(11, 4))
eixos[0].barh(ordem_shap, EXPLIC.loc[ordem_shap, "mean_abs_shap"], color="#1E88E5")
eixos[0].set_xlabel("|SHAP| médio (log-odds)"); eixos[0].set_title("Importância global", fontweight="bold")
jit = np.random.default_rng(SEED).uniform(-0.25, 0.25, len(SV))
for r, nome in enumerate(ordem_shap):
    j = PHENO_COLS.index(nome)
    sc = eixos[1].scatter(SV[:, j], np.full(len(SV), r) + jit, c=XZ[:, j], cmap="coolwarm", s=8, alpha=0.7, vmin=-2, vmax=2)
eixos[1].set_yticks(range(len(ordem_shap))); eixos[1].set_yticklabels(ordem_shap)
eixos[1].axvline(0, color="k", lw=0.8); eixos[1].set_xlabel("SHAP (log-odds de TDAH)")
eixos[1].set_title("Efeito por sujeito (cor = valor da variável)", fontweight="bold")
plt.colorbar(sc, ax=eixos[1], label="valor padronizado")
salvar("09_shap_fenotipos")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(SENS.index, SENS["AUC_media_sites"], yerr=SENS["AUC_dp_sites"], capsize=4, color="#7E57C2", alpha=0.85)
ax.axhline(0.5, color="gray", ls="--"); ax.set_ylim(0.35, 0.85)
ax.set_ylabel("AUC média entre sites (± DP)"); ax.set_title("Sensibilidade: subconjuntos de fenótipos", fontweight="bold")
salvar("10_sensibilidade_fenotipos")

## 14. Resumo final

In [ ]:
def interpretar():
    """Leitura automática dos resultados, restrita ao que os números sustentam."""
    linhas = []
    f = INFERENCIA.loc["pheno"]
    linhas.append(f"Fenótipos: AUC {f.AUC_media_sites:.3f} (IC95% {f.IC95_inf:.3f}-{f.IC95_sup:.3f}); "
                  f"p={f.p_bruto:.4f}; p corrigido (x{len(FAMILIA_CONFIRMATORIA)})={f.p_bonferroni:.4f}.")
    for m in ordem_mod:
        if m == "pheno":
            continue
        r = INFERENCIA.loc[m]
        sinal = "IC inclui 0,5" if r.IC95_inf <= 0.5 <= r.IC95_sup else ("IC acima de 0,5" if r.IC95_inf > 0.5 else "IC abaixo de 0,5")
        efeito = "pior" if r.delta_IC_sup < 0 else ("melhor" if r.delta_IC_inf > 0 else "sem diferença detectável")
        linhas.append(f"{r.rotulo}: AUC {r.AUC_media_sites:.3f} ({sinal}); contra fenótipos: {efeito} "
                      f"(delta {r.delta_vs_fenotipos:+.3f}, IC95% {r.delta_IC_inf:+.3f} a {r.delta_IC_sup:+.3f}).")
    return linhas


LEITURA = interpretar()
print("\n".join(LEITURA))

resumo = dict(
    n=int(len(y_all)), sites=sites, n_por_site={g: int((groups == g).sum()) for g in sites},
    inferencia=INFERENCIA.reset_index().to_dict(orient="records"),
    explicabilidade=EXPLIC.reset_index().to_dict(orient="records"),
    sensibilidade=SENS.reset_index().to_dict(orient="records"),
    poder=PODER.reset_index().to_dict(orient="records"),
    leitura=LEITURA,
    versoes={"numpy": np.__version__, "pandas": pd.__version__, "shap": None if shap is None else shap.__version__},
)
with open(f"{OUT_DIR}/resumo_final.json", "w", encoding="utf-8") as fh:
    json.dump(resumo, fh, indent=2, ensure_ascii=False, default=float)
shutil.make_archive(f"{OUT_DIR}/pacote_resultados", "zip", OUT_DIR, "figuras")
print(f"\nArquivos salvos em: {OUT_DIR}")